1. Sử dụng bộ data CIFAR -> có thể sử dụng dataloader hoặc các phương pháp đọc khác của PyTorch để tải dữ liệu.
2. Phân tách thành 2 hàm chính:
    - Hàm huấn luyện (train): Chịu trách nhiệm huấn luyện mô hình trên tập dữ liệu huấn luyện.
    - Hàm dự đoán (predict): Sử dụng mô hình đã được huấn luyện để dự đoán kết quả
3. Sử dụng phương pháp KNN với giá trị K = 5 hoặc 7 để thực thi quá trình phân lớp ảnh theo 2 hàm train và predict ở trên.
4. Sử dụng mô hình Linear Classifier để huấn luyện mô hình với bộ dữ liệu trên. Chọn loss là Softmax (có thể kết hợp với Cross-Entropy hoặc KL)
5. Sử dụng mô hình SVM để huấn luyện với bộ dữ liệu ở trên (kết hợp với SVM Loss)
6. Đưa ra bảng số liệu tổng hợp đánh giá accuracy, F1, precision cho 3 mô hình trên

# 1. Load data

In [1]:
import os
import pickle
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

def unpickle(file_path):
    with open(file_path, 'rb') as f:
        data = pickle.load(f, encoding='bytes')
    return data

def resolve_cifar100_root(data_dir):
    train_path = os.path.join(data_dir, 'train')
    test_path = os.path.join(data_dir, 'test')
    if os.path.exists(train_path) and os.path.exists(test_path):
        return data_dir

    nested_dir = os.path.join(data_dir, 'cifar-100-python')
    nested_train = os.path.join(nested_dir, 'train')
    nested_test = os.path.join(nested_dir, 'test')
    if os.path.exists(nested_train) and os.path.exists(nested_test):
        return nested_dir

    raise FileNotFoundError(
        f"Cannot find CIFAR-100 binary files in {data_dir}. "
        "Expected train/test directly or inside cifar-100-python/."
    )

def load_cifar100_data(data_dir):
    resolved_dir = resolve_cifar100_root(data_dir)
    train_data = unpickle(os.path.join(resolved_dir, 'train'))
    test_data = unpickle(os.path.join(resolved_dir, 'test'))

    Xtr = train_data[b'data'].astype(np.float32) / 255.0
    ytr = np.array(train_data[b'fine_labels'], dtype=np.int64)

    Xte = test_data[b'data'].astype(np.float32) / 255.0
    yte = np.array(test_data[b'fine_labels'], dtype=np.int64)

    return Xtr, ytr, Xte, yte

In [2]:
DATA_DIR = r"D:/Study/School/Deep learning/lab1/cifar-100-python"
Xtr, ytr, Xte, yte = load_cifar100_data(DATA_DIR)

print(f"Training data shape: {Xtr.shape}, Training labels shape: {ytr.shape}")
print(f"Testing data shape: {Xte.shape}, Testing labels shape: {yte.shape}")
print(f"Number of classes: {len(np.unique(ytr))}")

C:\Users\LEGION\AppData\Local\Temp\ipykernel_27840\1079589214.py:15: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data = pickle.load(f, encoding='bytes')


Training data shape: (50000, 3072), Training labels shape: (50000,)
Testing data shape: (10000, 3072), Testing labels shape: (10000,)
Number of classes: 100


# 2. Preprocess + Define train/predict functions

Để chạy nhanh trong notebook, ta lấy mẫu một phần dữ liệu (có stratify theo nhãn) cho cả train và test.

In [5]:
RANDOM_STATE = 42
MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 1000

def sample_dataset(X, y, max_samples, random_state=42):
    if len(y) <= max_samples:
        return X, y

    _, X_sample, _, y_sample = train_test_split(
        X, y,
        test_size=max_samples,
        stratify=y,
        random_state=random_state
    )
    return X_sample, y_sample

Xtr_sub, ytr_sub = sample_dataset(Xtr, ytr, MAX_TRAIN_SAMPLES, RANDOM_STATE)
Xte_sub, yte_sub = sample_dataset(Xte, yte, MAX_TEST_SAMPLES, RANDOM_STATE)

print(f"Sampled train shape: {Xtr_sub.shape}, labels: {ytr_sub.shape}")
print(f"Sampled test shape: {Xte_sub.shape}, labels: {yte_sub.shape}")

def train_knn(X_train, y_train, k=5):
    model = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    model.fit(X_train, y_train)
    return model

def predict_knn(model, X_test):
    return model.predict(X_test)

def train_linear_softmax(X_train, y_train):
    model = SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=1e-4,
        learning_rate='optimal',
        max_iter=100,
        tol=1e-3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model

def predict_linear_softmax(model, X_test):
    return model.predict(X_test)

def train_svm_with_hinge_loss(X_train, y_train):
    model = SGDClassifier(
        loss='hinge',
        penalty='l2',
        alpha=1e-4,
        learning_rate='optimal',
        max_iter=100,
        tol=1e-3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model

def predict_svm(model, X_test):
    return model.predict(X_test)

def evaluate_model(model_name, y_true, y_pred, train_seconds):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'Train time (s)': train_seconds
    }

Sampled train shape: (5000, 3072), labels: (5000,)
Sampled test shape: (1000, 3072), labels: (1000,)


# 3. Train + Predict + Evaluate 3 models

Thực hiện lần lượt KNN (K=5), Linear Classifier (Softmax/Cross-Entropy), và SVM; sau đó tổng hợp Accuracy, Precision, F1.

In [6]:
results = []

# KNN (K = 5)
start = time.time()
knn_model = train_knn(Xtr_sub, ytr_sub, k=5)
knn_train_time = time.time() - start
knn_pred = predict_knn(knn_model, Xte_sub)
results.append(evaluate_model('KNN (k=5)', yte_sub, knn_pred, knn_train_time))

# Linear Classifier with Softmax (log loss)
start = time.time()
softmax_model = train_linear_softmax(Xtr_sub, ytr_sub)
softmax_train_time = time.time() - start
softmax_pred = predict_linear_softmax(softmax_model, Xte_sub)
results.append(evaluate_model('Linear Softmax', yte_sub, softmax_pred, softmax_train_time))

# SVM loss (hinge)
start = time.time()
svm_model = train_svm_with_hinge_loss(Xtr_sub, ytr_sub)
svm_train_time = time.time() - start
svm_pred = predict_svm(svm_model, Xte_sub)
results.append(evaluate_model('SVM (hinge loss)', yte_sub, svm_pred, svm_train_time))

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
results_df

d:\Study\School\Data mining\DNHT\Course2026_DataMining\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


,Model,Accuracy,Precision,F1,Train time (s)
0,Linear Softmax,0.089,0.116448,0.076943,53.396003
1,SVM (hinge loss),0.078,0.106625,0.069418,18.181553
2,KNN (k=5),0.070,0.089615,0.063902,0.035007
